# Iterative bit-GRPO from the true-0.86 base — consolidated experiment

One notebook, one run: the **design of every GRPO iteration** + the results, in one place.

**Goal.** Can iterative on-policy GRPO push the LB past the 0.86 base on `bit_manipulation`
(the largest, mid-accuracy category) without regressing the rest?

**Base.** `output/kienngx-086-renamed` — the lossless kienngx adapter with keys renamed
`base_model.model.model.*` -> `base_model.model.backbone.*` (now identical key-set to the
Nemotron-H modules, so HF-PEFT loads it correctly AND it file-submits at **LB 0.86**). The
earlier `model.model`-keyed copy mis-loaded -> KL exploded (kl_loss ~17000) -> collapse; the
rename fixed both training and submission.

## Design (identical across all iterations unless noted)

| Component | Setting |
|---|---|
| Algorithm | GRPO, group-relative ± advantage, Schulman-k3 token KL |
| Warmstart (iter-1) | `kienngx-086-renamed` (true 0.86); iter-t warmstarts from `policy_iter{t}` |
| KL anchor (ref) | the warmstart adapter (KL-to-warmstart), β = 0.02 |
| ref_logp | precomputed by a **separate low-util vLLM pass** (`score_ref_logp.py`, gpu_util 0.55) to avoid the prompt_logprobs OOM at the generation engine's 0.85 util |
| Rollout | G=8, T=1.0, top_p=1.0, max_tokens=7680, max_model_len=8192; **multi-node** (2×TP4 over RoCE, each node a pool shard), max_num_seqs=64 |
| Pool | `bit_grpo_train96.jsonl` = 96 bit problems (64 greedy-fail + 32 greedy-correct), disjoint from the holdout |
| Train | β=0.02, lr=5e-6, batch_size=1, grad_acc=8, **10 optimizer steps / iteration**, max_length=8192, gradient-checkpointing on; 2-node 8-way FSDP FULL_SHARD |
| Eval (per iter) | T=0 greedy on a **disjoint 100-problem bit holdout** (50 greedy-fail + 50 greedy-correct); net = recover (fail→right) − regress (right→wrong) |
| Iterations | each = resample from the current policy → 10 near-on-policy steps → eval; chain self-resubmits one iter/job (HPC: short 3h windows, checkpoint each) |

In [ ]:
import json, glob, os
import matplotlib.pyplot as plt

S = '/scratch/jwj/NemotronFT/training/rl/runs/bit_grpo_iter_086'

# --- per-iteration holdout net-gain (vs the true-0.86 warmstart) ---
iters, net, overall, recov, regr = [], [], [], [], []
for k in range(1, 8):
    hp = f'{S}/policy_iter{k}/holdout.json'
    if not os.path.exists(hp):
        continue
    d = json.load(open(hp))
    iters.append(k); net.append(d['net']); overall.append(d['trained_overall'])
    recov.append(d['recover']); regr.append(d['regress'])
warm_overall = json.load(open(f'{S}/policy_iter1/holdout.json'))['warm_overall'] if iters else None

print('iter | net | overall | recover | regress')
for i in range(len(iters)):
    print(f'  {iters[i]}  | {net[i]:+d}  |  {overall[i]:.2f}   |    {recov[i]}    |   {regr[i]}')
print(f'warmstart (true 0.86) holdout overall = {warm_overall}')

# --- rollout collapse-watch per iteration ---
print('\nrollout collapse-watch:')
for k in range(0, 7):
    sp = f'{S}/rollouts/iter{k}.stats.json'
    if os.path.exists(sp):
        r = json.load(open(sp))
        print(f"  iter{k}: reward_mean={r['reward_mean']:.3f} frac_mixed={r['frac_mixed_groups']:.2f} "
              f"resp_len={r['resp_len_mean']:.0f} distinct={r['distinct_ratio']:.2f} trunc={r['frac_truncated']:.2f}")

In [ ]:
# Net-gain + holdout overall vs iteration (the 'dip then climb' question)
if iters:
    fig, ax = plt.subplots(1, 2, figsize=(11, 4))
    ax[0].plot(iters, net, 'o-'); ax[0].axhline(0, ls='--', c='gray')
    ax[0].set_xlabel('iteration'); ax[0].set_ylabel('holdout net (recover−regress)'); ax[0].set_title('net-gain vs iter')
    ax[1].plot(iters, overall, 'o-', label='trained')
    if warm_overall is not None:
        ax[1].axhline(warm_overall, ls='--', c='r', label='warmstart 0.86')
    ax[1].set_xlabel('iteration'); ax[1].set_ylabel('holdout overall acc'); ax[1].set_title('overall vs iter'); ax[1].legend()
    plt.tight_layout(); plt.show()

## Results & Leaderboard

| Artifact | holdout (net / overall) | **LB** |
|---|---|---|
| `kienngx-086-renamed` (base) | — | **0.86** |
| bit-only GRPO iter-1 | +1 / 0.59 | **0.85** |
| bit-only GRPO iter-3 | +2 / 0.61 | **0.85** |
| bit-only GRPO iter-5 | +2 / 0.62 (highest) | **0.84** |
| **multi-category GRPO iter-1** | 0 / 0.58 | **0.86** (held — no regression) |

Multi-category pool = bit + eqn_deduce (gradient-bearing) + cipher/gravity/numeral/unit
(near-saturated, ~no gradient but KL-pinned); cryptarithm/eqn_guess excluded (pass@k≈0 → no signal).

**Findings.**
1. **KL stays healthy** with the correctly-keyed base (kl_loss 0–24 vs the ~17000 explosion when the key-set mismatched the modules). Rollout diversity holds (frac_mixed ~0.47–0.51, distinct ~0.78, no length collapse: resp_len ~6700, trunc ~0.02).
2. **★ The bit-holdout is ANTI-CORRELATED with the LB.** Across iterations the bit-holdout *climbs* (overall 0.59→0.61→0.62) while the LB *falls* (0.85→0.85→0.84). Optimizing the 100-problem bit-holdout actively *hurts* the full 9500-problem hidden metric. This is the clearest demonstration of the proxy-holdout trap + prior-coupling: bit-only GRPO over-optimizes bit at the expense of the rest, and the bit-holdout cannot see that cost.
3. **bit-only GRPO progressively REGRESSES on the LB: 0.86 → 0.85 → 0.84** as it iterates. More iterations = worse.
4. **Multi-category GRPO holds LB 0.86 (no regression)** — strictly better than bit-only. Including the saturated categories in the pool + KL anchor pins them in place, so the policy can't over-optimize bit and drift. It still can't *exceed* 0.86 (bit/eqn_deduce have no learnable headroom past the base), but it avoids the bit-only collapse.
5. **Net: GRPO from the true-0.86 base, strongest config (correct keys, healthy KL, multi-node, on-policy) = LB 0.86 at best (multi-category, held), and bit-only actively regresses to 0.84.** Consistent with RFT (same prior-coupling mechanism). The holdout-vs-LB anti-correlation is the methodological headline.